# Titanic Dataset: Data Acquisition, Cleaning, and Preprocessing

This notebook explains the pipeline. Run from the project root or change paths if using Colab.

## 1. Install dependencies
Run `pip install -r requirements.txt` in the project terminal if required.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
DATA_URL = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv'
raw_path = ROOT / 'data' / 'raw' / 'titanic.csv'
raw_path.parent.mkdir(parents=True, exist_ok=True)
if not raw_path.exists():
    df_download = pd.read_csv(DATA_URL)
    df_download.to_csv(raw_path, index=False)
raw = pd.read_csv(raw_path)
df = raw.copy()
print('Shape:', df.shape)
display(df.head())

## 2. Initial quality checks
Look at types, summaries, missingness, and duplicate rows before changing anything.

In [ ]:
display(df.info())
display(df.describe(include='all').T)
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame('missing_count'))
print('Exact duplicate rows:', df.duplicated().sum())

## 3. Normalize text and validate values
Whitespace and empty strings can create false categories. Domain checks help distinguish invalid values from rare but valid observations.

In [ ]:
text_cols = df.select_dtypes(include=['object', 'string']).columns
for col in text_cols:
    df[col] = df[col].map(lambda x: x.strip() if isinstance(x, str) else x)
    df[col] = df[col].replace('', np.nan)
if 'age' in df: print('Negative ages:', (df.age < 0).sum())
if 'fare' in df: print('Negative fares:', (df.fare < 0).sum())
if 'survived' in df: print('Invalid survival labels:', (~df.survived.isin([0,1]) & df.survived.notna()).sum())

## 4. Visualize missingness and outliers
The IQR rule flags observations outside Q1 − 1.5×IQR and Q3 + 1.5×IQR. These are review flags, not proof of errors.

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
if len(missing):
    missing.plot(kind='bar', figsize=(10,4), title='Missing values by column')
    plt.ylabel('Count'); plt.tight_layout(); plt.show()
for col in ['age', 'fare']:
    if col in df:
        sns.boxplot(x=df[col]); plt.title(f'{col.title()} distribution'); plt.show()
        s = df[col].dropna(); q1, q3 = s.quantile([.25,.75]); iqr = q3-q1
        count = ((s < q1-1.5*iqr) | (s > q3+1.5*iqr)).sum()
        print(col, 'IQR outlier count:', int(count))

## 5. Clean missing values and duplicate rows
This baseline removes exact duplicate rows only if any exist. Numeric columns use the median; categorical columns use the mode, except deck, where missingness is represented as Unknown. Consider whether missingness itself carries meaning.

In [ ]:
clean = df.drop_duplicates().copy()
for col in clean.select_dtypes(include=np.number).columns:
    if clean[col].isna().any(): clean[col] = clean[col].fillna(clean[col].median())
for col in clean.select_dtypes(include=['object','string','category']).columns:
    if clean[col].isna().any():
        if col == 'deck': clean[col] = clean[col].astype('object').fillna('Unknown')
        else:
            mode = clean[col].mode(dropna=True)
            clean[col] = clean[col].fillna(mode.iloc[0] if len(mode) else 'Unknown')
print('Before:', df.shape, 'After:', clean.shape)
print('Remaining missing cells:', int(clean.isna().sum().sum()))

## 6. Save outputs
Keep raw and processed data separate so that the workflow is auditable.

In [ ]:
out = ROOT / 'data' / 'processed' / 'titanic_cleaned.csv'
out.parent.mkdir(parents=True, exist_ok=True)
clean.to_csv(out, index=False)
print('Saved:', out)

## Reflection
Imputation preserves rows but can distort distributions and correlations. IQR flags do not automatically justify removal. For predictive modeling, split the data first and fit imputers/scalers only on training data to prevent leakage. Record actual outputs from your run in the report.